# Module 1, Notebook 4: Advanced Python

## For Senior Swift Engineers

This notebook covers Python's most powerful abstractions: first-class functions, decorators, generators, context managers, and type hints. Many of these concepts have Swift analogs, but several (especially decorators and generators) work quite differently.

---

## Swift vs Python Quick Reference

| Concept | Swift | Python |
|---|---|---|
| First-class functions | Functions are first-class types | Functions are first-class objects |
| Closures / Lambdas | `{ (params) -> Return in body }` | `lambda params: expr` (single expression only) |
| Decorators | No direct equivalent (use protocol extensions, property wrappers) | `@decorator` syntax; wraps functions/classes |
| Generators | `Sequence`/`IteratorProtocol` conformance | `yield` keyword in functions |
| Context managers | `defer` for cleanup; no `with` equivalent | `with` statement + `__enter__`/`__exit__` |
| Type hints | Strong static typing (enforced at compile time) | Optional annotations (not enforced at runtime) |
| `Optional` | `Optional<T>` / `T?` (built into type system) | `Optional[T]` from `typing` (just a hint) |
| Generics | `func foo<T: Protocol>(x: T)` | `TypeVar`, `Generic[T]` from `typing` |
| `@property` | Computed properties (`var x: Int { get { } }`) | `@property` decorator on methods |
| Caching | No built-in (use libraries) | `@functools.lru_cache` |
| Static methods | `static func` / `class func` | `@staticmethod` / `@classmethod` |

---

## 1. First-Class Functions and Lambdas

In Swift, you are already comfortable with the idea that functions and closures are first-class citizens. Python is the same: functions are objects. You can assign them to variables, pass them as arguments, return them from other functions, and store them in data structures.

### Swift comparison

```swift
// Swift: functions are first-class
func greet(_ name: String) -> String {
    return "Hello, \(name)!"
}

let greeter: (String) -> String = greet
print(greeter("Daniel"))  // Hello, Daniel!

// Swift closures
let double = { (x: Int) -> Int in x * 2 }
let numbers = [1, 2, 3].map { $0 * 2 }  // trailing closure syntax
```

In [ ]:
# Python: functions are first-class objects

def greet(name: str) -> str:
    return f"Hello, {name}!"

# Assign function to a variable (no parentheses = reference, not call)
greeter = greet
print(greeter("Daniel"))  # Hello, Daniel!
print(type(greeter))       # <class 'function'>

# Functions have attributes — they are full objects
print(greeter.__name__)    # greet
print(greeter.__doc__)     # None (we didn't add a docstring)

In [ ]:
# Passing functions as arguments — just like Swift's higher-order functions

def apply_operation(x: int, y: int, operation):
    """Apply a function to two numbers."""
    return operation(x, y)

def add(a, b):
    return a + b

def multiply(a, b):
    return a * b

print(apply_operation(3, 4, add))       # 7
print(apply_operation(3, 4, multiply))  # 12

In [ ]:
# Lambdas — Python's anonymous functions
# KEY DIFFERENCE from Swift closures: lambdas can only contain a SINGLE expression.
# No statements, no multi-line bodies. For anything complex, use a named function.

# Swift equivalent: { (x: Int) -> Int in x * 2 }
double = lambda x: x * 2
print(double(5))  # 10

# Lambdas shine with higher-order functions
numbers = [1, 2, 3, 4, 5]

# map, filter in Python — compare to Swift's .map { } and .filter { }
doubled = list(map(lambda x: x * 2, numbers))
evens = list(filter(lambda x: x % 2 == 0, numbers))

print(f"Doubled: {doubled}")  # [2, 4, 6, 8, 10]
print(f"Evens: {evens}")      # [2, 4]

# But Pythonic style prefers list comprehensions over map/filter:
doubled_pythonic = [x * 2 for x in numbers]
evens_pythonic = [x for x in numbers if x % 2 == 0]
print(f"Doubled (comprehension): {doubled_pythonic}")
print(f"Evens (comprehension): {evens_pythonic}")

In [ ]:
# Returning functions from functions (closures that capture state)

# Swift equivalent:
# func makeMultiplier(_ factor: Int) -> (Int) -> Int {
#     return { number in number * factor }
# }

def make_multiplier(factor: int):
    """Return a function that multiplies by the given factor."""
    def multiplier(number: int) -> int:
        return number * factor  # 'factor' is captured from enclosing scope
    return multiplier

triple = make_multiplier(3)
print(triple(10))  # 30
print(triple(7))   # 21

# Sorting with key functions — very common in Python
students = [("Alice", 88), ("Bob", 95), ("Charlie", 72)]
students_by_grade = sorted(students, key=lambda s: s[1], reverse=True)
print(students_by_grade)  # [('Bob', 95), ('Alice', 88), ('Charlie', 72)]

---

## 2. Decorators

**This concept does not exist in Swift.** Swift has property wrappers (`@State`, `@Published`, `@ObservableObject`) which look similar syntactically, but decorators are fundamentally different.

A **decorator** is a function that takes a function as input and returns a new function (usually wrapping the original). The `@decorator` syntax is just syntactic sugar.

### The core idea

```python
@my_decorator
def my_function():
    pass

# is EXACTLY equivalent to:
def my_function():
    pass
my_function = my_decorator(my_function)
```

That is it. A decorator replaces a function with whatever the decorator returns.

In [ ]:
# Building a decorator from scratch

def shout(func):
    """A decorator that uppercases the return value of a function."""
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)  # call the original function
        return result.upper()           # modify the result
    return wrapper

@shout
def greet(name):
    return f"hello, {name}"

print(greet("Daniel"))  # HELLO, DANIEL

# What actually happened:
# greet = shout(greet)
# So greet is now the 'wrapper' function that shout returned.

In [ ]:
# Problem: the decorated function loses its identity
print(greet.__name__)  # 'wrapper' — not 'greet'!

# Solution: functools.wraps preserves the original function's metadata
import functools

def shout_fixed(func):
    """A decorator that uppercases the return value of a function."""
    @functools.wraps(func)  # <-- this preserves __name__, __doc__, etc.
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

@shout_fixed
def greet_v2(name):
    """Greet someone by name."""
    return f"hello, {name}"

print(greet_v2.__name__)  # 'greet_v2' — correct!
print(greet_v2.__doc__)   # 'Greet someone by name.' — correct!

In [ ]:
# A practical decorator: timing function execution
import functools
import time

def timer(func):
    """Decorator that prints execution time of a function."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f} seconds")
        return result
    return wrapper

@timer
def slow_sum(n):
    """Sum numbers the slow way."""
    return sum(range(n))

result = slow_sum(1_000_000)
print(f"Result: {result}")

In [ ]:
# Decorators with arguments
# When you see @decorator(args), you need an extra level of nesting:
# The outer function takes the decorator's arguments and returns the actual decorator.

import functools

def repeat(times: int):
    """Decorator that calls the function multiple times."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            results = []
            for _ in range(times):
                results.append(func(*args, **kwargs))
            return results
        return wrapper
    return decorator

@repeat(times=3)
def say_hello(name):
    print(f"Hello, {name}!")
    return f"greeted {name}"

results = say_hello("Daniel")
print(f"Results: {results}")

In [ ]:
# Common built-in decorators you'll encounter everywhere

class Temperature:
    """Demonstrates @staticmethod, @classmethod, and @property."""
    
    def __init__(self, celsius: float):
        self._celsius = celsius  # 'private' by convention
    
    # @property — makes a method behave like a read-only attribute
    # Swift equivalent: var fahrenheit: Double { celsius * 9/5 + 32 }
    @property
    def fahrenheit(self) -> float:
        return self._celsius * 9 / 5 + 32
    
    @property
    def celsius(self) -> float:
        return self._celsius
    
    # @celsius.setter — allows assignment to the property
    # Swift equivalent: set { } inside the computed property
    @celsius.setter
    def celsius(self, value: float):
        if value < -273.15:
            raise ValueError("Temperature below absolute zero!")
        self._celsius = value
    
    # @staticmethod — no access to instance or class
    # Swift equivalent: static func
    @staticmethod
    def is_boiling(temp_celsius: float) -> bool:
        return temp_celsius >= 100.0
    
    # @classmethod — receives the class as first argument (cls)
    # Swift equivalent: class func (or convenience init)
    @classmethod
    def from_fahrenheit(cls, fahrenheit: float) -> "Temperature":
        celsius = (fahrenheit - 32) * 5 / 9
        return cls(celsius)  # cls() so subclasses work correctly

# Usage
t = Temperature(100)
print(f"{t.celsius}C = {t.fahrenheit}F")  # 100C = 212.0F

# Property setter
t.celsius = 0
print(f"{t.celsius}C = {t.fahrenheit}F")  # 0C = 32.0F

# Static method
print(f"Is 100C boiling? {Temperature.is_boiling(100)}")  # True

# Class method as alternative constructor
t2 = Temperature.from_fahrenheit(72)
print(f"72F = {t2.celsius:.1f}C")  # 72F = 22.2C

In [ ]:
# @functools.lru_cache — automatic memoization
# No direct Swift equivalent (you'd build your own cache dict or use NSCache)

import functools

@functools.lru_cache(maxsize=128)
def fibonacci(n: int) -> int:
    """Compute the nth Fibonacci number with automatic memoization."""
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

# Without caching, this would be exponentially slow
print(f"fib(50) = {fibonacci(50)}")

# Inspect the cache
print(f"Cache info: {fibonacci.cache_info()}")

# Clear the cache if needed
fibonacci.cache_clear()

---

## 3. Generators and `yield`

Generators are Python's elegant solution for lazy sequences. Instead of building an entire list in memory, a generator produces values one at a time, on demand.

### Swift comparison

In Swift, you achieve lazy sequences by conforming to `Sequence` and `IteratorProtocol`:

```swift
struct CountUp: Sequence, IteratorProtocol {
    var current = 0
    let limit: Int
    
    mutating func next() -> Int? {
        guard current < limit else { return nil }
        defer { current += 1 }
        return current
    }
}
```

In Python, generators let you do the same thing with far less boilerplate.

In [ ]:
# A generator function: any function that uses 'yield' instead of 'return'

def count_up(limit: int):
    """Generate numbers from 0 to limit-1."""
    current = 0
    while current < limit:
        yield current      # pause here, produce this value
        current += 1       # resume here on next iteration
    # function ends → StopIteration is raised automatically

# Calling a generator function doesn't execute it — it returns a generator object
gen = count_up(5)
print(type(gen))   # <class 'generator'>

# Pull values one at a time with next()
print(next(gen))   # 0
print(next(gen))   # 1
print(next(gen))   # 2

# Or iterate with a for loop (most common)
for num in count_up(5):
    print(num, end=" ")  # 0 1 2 3 4
print()

In [ ]:
# Generator expressions — like list comprehensions, but lazy
# Use () instead of []

# List comprehension: builds entire list in memory
squares_list = [x ** 2 for x in range(1_000_000)]  # ~8MB of memory

# Generator expression: produces values on demand
squares_gen = (x ** 2 for x in range(1_000_000))   # ~120 bytes!

import sys
print(f"List size: {sys.getsizeof(squares_list):,} bytes")
print(f"Generator size: {sys.getsizeof(squares_gen)} bytes")

# You can still use the generator anywhere you'd use an iterable
total = sum(x ** 2 for x in range(1_000_000))  # no extra [] needed inside sum()
print(f"Sum of squares: {total:,}")

In [ ]:
# Memory efficiency demo: reading large files line by line
# This is one of the most practical uses of generators

def read_large_file(filepath: str):
    """Yield lines from a file without loading entire file into memory."""
    with open(filepath, "r") as f:
        for line in f:    # file objects are themselves iterators!
            yield line.strip()

# In practice you would do:
# for line in read_large_file("huge_logfile.txt"):
#     if "ERROR" in line:
#         process(line)

# The key insight: only one line is in memory at a time,
# even for a 10GB file.

In [ ]:
# Building a pipeline with generators
# Each stage lazily transforms data — like Unix pipes

def generate_numbers(n: int):
    """Stage 1: produce numbers."""
    for i in range(n):
        yield i

def filter_evens(numbers):
    """Stage 2: keep only even numbers."""
    for n in numbers:
        if n % 2 == 0:
            yield n

def square(numbers):
    """Stage 3: square each number."""
    for n in numbers:
        yield n ** 2

def take(iterable, count: int):
    """Stage 4: take first 'count' items."""
    for i, item in enumerate(iterable):
        if i >= count:
            return
        yield item

# Compose the pipeline — nothing executes until we iterate!
pipeline = take(square(filter_evens(generate_numbers(1000))), 5)

# Now pull values through the pipeline
result = list(pipeline)
print(result)  # [0, 4, 16, 36, 64]

# Only 10 numbers were generated (0-9), not all 1000!

In [ ]:
# itertools — the standard library's treasure trove for iterator operations
# Think of it as Python's equivalent of Swift's lazy sequence operations

import itertools

# itertools.chain — concatenate iterables (like Swift's + for sequences)
combined = itertools.chain([1, 2], [3, 4], [5, 6])
print(list(combined))  # [1, 2, 3, 4, 5, 6]

# itertools.islice — slice an iterator (like Swift's prefix/suffix)
first_five = itertools.islice(range(100), 5)
print(list(first_five))  # [0, 1, 2, 3, 4]

# itertools.groupby — group consecutive elements (like Dictionary(grouping:by:))
data = [("fruit", "apple"), ("fruit", "banana"), ("veg", "carrot"), ("veg", "pea")]
for key, group in itertools.groupby(data, key=lambda x: x[0]):
    print(f"{key}: {[item[1] for item in group]}")

# itertools.product — Cartesian product (like nested for loops)
colors = ["red", "blue"]
sizes = ["S", "M", "L"]
combos = list(itertools.product(colors, sizes))
print(f"Combinations: {combos}")

# itertools.count — infinite counter
counter = itertools.count(start=10, step=5)
print([next(counter) for _ in range(5)])  # [10, 15, 20, 25, 30]

---

## 4. Context Managers (`with` Statement)

Context managers guarantee that setup and cleanup code runs, even if an exception occurs. The closest Swift analog is `defer`, but context managers are more structured.

### Swift comparison

```swift
// Swift: defer for cleanup
func processFile() {
    let file = openFile("data.txt")
    defer { file.close() }  // runs when scope exits
    // ... use file ...
}  // file.close() called here
```

```python
# Python: with statement
with open("data.txt") as file:
    # ... use file ...
# file.close() called here automatically
```

The key difference: `defer` is a general cleanup mechanism, while `with` is a protocol (`__enter__`/`__exit__`) that any class can implement.

In [ ]:
# The with statement in action

# Most common use: file handling
# The file is guaranteed to be closed, even if an exception occurs
import tempfile
import os

# Create a temporary file for demonstration
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write("Hello from context manager!\n")
    f.write("Line 2\n")
    temp_path = f.name

# Read it back
with open(temp_path, 'r') as f:
    content = f.read()
    print(content)

# Clean up
os.unlink(temp_path)

# After the with block, f.closed is True
print(f"File closed? {f.closed}")  # True

In [ ]:
# Building a context manager class: implement __enter__ and __exit__

import time

class Timer:
    """Context manager that times a block of code."""
    
    def __enter__(self):
        """Called when entering the 'with' block. Return value is bound to 'as'."""
        self.start = time.perf_counter()
        return self  # the object bound to 'as timer'
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """Called when exiting the 'with' block, even if an exception occurred.
        
        Args:
            exc_type: Exception class (or None if no exception)
            exc_val: Exception instance (or None)
            exc_tb: Traceback (or None)
        
        Returns:
            True to suppress the exception, False (or None) to propagate it.
        """
        self.elapsed = time.perf_counter() - self.start
        print(f"Elapsed: {self.elapsed:.4f} seconds")
        return False  # don't suppress exceptions

# Usage
with Timer() as t:
    total = sum(range(1_000_000))
    print(f"Total: {total:,}")

# You can access the timer after the block too
print(f"It took {t.elapsed:.4f}s")

In [ ]:
# Easier way: @contextmanager decorator from contextlib
# Write a generator function instead of a class

from contextlib import contextmanager
import time

@contextmanager
def timer(label: str = "Block"):
    """Context manager that times a block of code."""
    start = time.perf_counter()
    try:
        yield start  # everything before yield = __enter__, yielded value = 'as' target
    finally:
        elapsed = time.perf_counter() - start  # everything after yield = __exit__
        print(f"{label}: {elapsed:.4f} seconds")

# Usage — same as before, but much less code to write
with timer("Summation"):
    total = sum(range(1_000_000))
    print(f"Total: {total:,}")

In [ ]:
# Practical context manager: temporary directory change

from contextlib import contextmanager
import os

@contextmanager
def change_directory(path: str):
    """Temporarily change the working directory.
    
    Like Swift's defer pattern, but more structured.
    """
    original = os.getcwd()
    try:
        os.chdir(path)
        yield
    finally:
        os.chdir(original)  # always restore, even on exception

print(f"Before: {os.getcwd()}")

with change_directory("/tmp"):
    print(f"Inside: {os.getcwd()}")

print(f"After: {os.getcwd()}")

---

## 5. Type Hints Deep Dive

Coming from Swift, you expect a strong, expressive type system. Python's type hints provide similar expressiveness, but with one critical difference: **they are not enforced at runtime**. They exist for:

1. **Documentation** — readers (and you) understand the code faster
2. **IDE support** — autocompletion, inline errors
3. **Static analysis** — tools like `mypy` catch type errors before runtime

### Swift comparison

```swift
// Swift: types are enforced at compile time
func greet(name: String) -> String { ... }  // MUST return String
```

```python
# Python: type hints are suggestions, not enforced
def greet(name: str) -> str: ...  # CAN return anything at runtime
```

In [ ]:
# Basic type hints (Python 3.10+ syntax)
# These use the modern, cleaner syntax — no need to import from typing

# Simple types
name: str = "Daniel"
age: int = 30
height: float = 5.11
is_developer: bool = True

# Collection types (Python 3.9+: use built-in names instead of typing.List, etc.)
numbers: list[int] = [1, 2, 3]
name_to_age: dict[str, int] = {"Alice": 30, "Bob": 25}
unique_ids: set[int] = {1, 2, 3}
coordinate: tuple[float, float] = (3.14, 2.71)
mixed_tuple: tuple[str, int, bool] = ("hello", 42, True)
variable_tuple: tuple[int, ...] = (1, 2, 3, 4)  # variable-length tuple of ints

# Function type hint
def process_items(items: list[str], transform: callable) -> list[str]:
    return [transform(item) for item in items]

print(process_items(["hello", "world"], str.upper))

In [ ]:
# Optional and Union types

# Swift: var name: String? = nil
# Python 3.10+: use X | None
def find_user(user_id: int) -> str | None:
    """Return username or None if not found."""
    users = {1: "Alice", 2: "Bob"}
    return users.get(user_id)

# Union types (Swift doesn't have union types — you'd use enums or protocols)
# Python 3.10+: use X | Y
def parse_id(value: str | int) -> int:
    """Accept either a string or int ID."""
    if isinstance(value, str):
        return int(value)
    return value

print(find_user(1))     # Alice
print(find_user(999))   # None
print(parse_id("42"))   # 42
print(parse_id(42))     # 42

# For Python < 3.10, use Optional and Union from typing:
from typing import Optional, Union
# Optional[str] is equivalent to str | None
# Union[str, int] is equivalent to str | int

In [ ]:
# Callable type hints — for function parameters
# Swift: (Int, Int) -> Int

from collections.abc import Callable

# A function that takes two ints and returns an int
def apply_op(x: int, y: int, op: Callable[[int, int], int]) -> int:
    return op(x, y)

print(apply_op(3, 4, lambda a, b: a + b))  # 7

# Type alias — clean up complex types
# Swift: typealias Handler = (Request) -> Response
type Handler = Callable[[str], str]  # Python 3.12+ syntax

# For older Python:
# Handler = Callable[[str], str]

def process(text: str, handler: Handler) -> str:
    return handler(text)

print(process("hello", str.upper))  # HELLO

In [ ]:
# Generics with TypeVar — Python's version of Swift generics

# Swift:
# func first<T>(of array: [T]) -> T? {
#     return array.first
# }

from typing import TypeVar

T = TypeVar('T')

def first(items: list[T]) -> T | None:
    """Return the first item or None."""
    return items[0] if items else None

# The type checker understands that first([1,2,3]) returns int | None
print(first([1, 2, 3]))       # 1
print(first(["a", "b"]))      # 'a'
print(first([]))               # None

# Bounded TypeVar — like Swift's <T: Comparable>
from typing import TypeVar

# Python 3.12+ syntax:
# def max_item[T: (int, float)](items: list[T]) -> T: ...

# Compatible syntax:
Numeric = TypeVar('Numeric', int, float)

def max_item(items: list[Numeric]) -> Numeric:
    return max(items)

print(max_item([3, 1, 4, 1, 5]))  # 5

In [ ]:
# Generic classes — like Swift's generic types

# Swift:
# class Stack<Element> {
#     private var items: [Element] = []
#     func push(_ item: Element) { items.append(item) }
#     func pop() -> Element? { items.popLast() }
# }

from typing import TypeVar, Generic

T = TypeVar('T')

class Stack(Generic[T]):
    """A generic stack — type-safe with mypy."""
    
    def __init__(self) -> None:
        self._items: list[T] = []
    
    def push(self, item: T) -> None:
        self._items.append(item)
    
    def pop(self) -> T:
        if not self._items:
            raise IndexError("pop from empty stack")
        return self._items.pop()
    
    def peek(self) -> T | None:
        return self._items[-1] if self._items else None
    
    def __len__(self) -> int:
        return len(self._items)
    
    def __repr__(self) -> str:
        return f"Stack({self._items})"

# Usage
int_stack: Stack[int] = Stack()
int_stack.push(1)
int_stack.push(2)
int_stack.push(3)
print(int_stack)          # Stack([1, 2, 3])
print(int_stack.pop())    # 3
print(int_stack.peek())   # 2
print(len(int_stack))     # 2

In [ ]:
# Protocol (structural typing) — like Swift protocols but based on structure
# Python 3.8+ with typing.Protocol

from typing import Protocol, runtime_checkable

# Swift equivalent:
# protocol Drawable {
#     func draw() -> String
# }

@runtime_checkable
class Drawable(Protocol):
    def draw(self) -> str: ...

class Circle:
    def draw(self) -> str:
        return "Drawing a circle"

class Square:
    def draw(self) -> str:
        return "Drawing a square"

class NotDrawable:
    pass

def render(shape: Drawable) -> None:
    print(shape.draw())

# KEY DIFFERENCE from Swift:
# Circle/Square don't need to explicitly declare they implement Drawable.
# Python uses 'structural typing' (duck typing) — if it has a draw() method, it works.

render(Circle())  # Drawing a circle
render(Square())  # Drawing a square

# runtime_checkable lets you use isinstance()
print(isinstance(Circle(), Drawable))      # True
print(isinstance(NotDrawable(), Drawable))  # False

---

## Exercises

### Exercise 1: Write a Timer Decorator

Create a `@timed` decorator that:
1. Measures how long a function takes to execute
2. Stores the elapsed time as an attribute on the function (`func.last_elapsed`)
3. Optionally logs to a provided logger function (decorator with arguments)
4. Preserves the original function's metadata with `@functools.wraps`

In [ ]:
# Exercise 1: Your solution here

import functools
import time

def timed(func=None, *, logger=None):
    """Decorator that times function execution.
    
    Can be used with or without arguments:
        @timed
        def foo(): ...
        
        @timed(logger=print)
        def bar(): ...
    """
    # YOUR CODE HERE
    pass

# Test it:
# @timed
# def slow_add(a, b):
#     time.sleep(0.1)
#     return a + b

# result = slow_add(3, 4)
# print(f"Result: {result}")
# print(f"Elapsed: {slow_add.last_elapsed:.4f}s")

In [ ]:
# Exercise 1: Solution

import functools
import time

def timed(func=None, *, logger=None):
    """Decorator that times function execution.
    
    Can be used with or without arguments:
        @timed
        def foo(): ...
        
        @timed(logger=print)
        def bar(): ...
    """
    # If called without arguments, func is the decorated function
    # If called with arguments, func is None and we return a decorator
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            start = time.perf_counter()
            result = fn(*args, **kwargs)
            elapsed = time.perf_counter() - start
            wrapper.last_elapsed = elapsed
            
            message = f"{fn.__name__} took {elapsed:.4f}s"
            if logger:
                logger(message)
            
            return result
        
        wrapper.last_elapsed = 0.0
        return wrapper
    
    if func is not None:
        # @timed without parentheses
        return decorator(func)
    # @timed() or @timed(logger=print)
    return decorator

# Test without arguments
@timed
def slow_add(a, b):
    """Add two numbers slowly."""
    time.sleep(0.1)
    return a + b

result = slow_add(3, 4)
print(f"Result: {result}")
print(f"Elapsed: {slow_add.last_elapsed:.4f}s")
print(f"Name preserved: {slow_add.__name__}")
print(f"Doc preserved: {slow_add.__doc__}")

print()

# Test with logger argument
log_messages = []

@timed(logger=lambda msg: log_messages.append(msg))
def fast_multiply(a, b):
    return a * b

fast_multiply(3, 4)
fast_multiply(5, 6)
print(f"Log messages: {log_messages}")

### Exercise 2: Build a Generator Pipeline

Create a pipeline of generators that processes a stream of log entries:
1. `generate_logs(n)` — yield `n` fake log entries as dicts with `timestamp`, `level`, and `message`
2. `filter_by_level(logs, level)` — yield only logs matching the given level
3. `format_logs(logs)` — yield formatted strings
4. `take(iterable, n)` — yield at most `n` items

Chain them together and verify memory efficiency.

In [ ]:
# Exercise 2: Your solution here

import random
from datetime import datetime, timedelta

# YOUR CODE HERE
pass

In [ ]:
# Exercise 2: Solution

import random
from datetime import datetime, timedelta

def generate_logs(n: int):
    """Stage 1: yield n fake log entries."""
    levels = ["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"]
    messages = [
        "User logged in",
        "Database query executed",
        "Cache miss",
        "Connection timeout",
        "Disk space low",
        "Invalid request",
        "Service started",
        "Retry attempt",
    ]
    base_time = datetime(2025, 1, 1)
    
    for i in range(n):
        yield {
            "timestamp": base_time + timedelta(seconds=i * 30),
            "level": random.choice(levels),
            "message": random.choice(messages),
        }

def filter_by_level(logs, level: str):
    """Stage 2: yield only logs matching the given level."""
    for log in logs:
        if log["level"] == level:
            yield log

def format_logs(logs):
    """Stage 3: yield formatted log strings."""
    for log in logs:
        ts = log["timestamp"].strftime("%Y-%m-%d %H:%M:%S")
        yield f"[{ts}] {log['level']}: {log['message']}"

def take(iterable, n: int):
    """Stage 4: yield at most n items."""
    for i, item in enumerate(iterable):
        if i >= n:
            return
        yield item

# Compose the pipeline
random.seed(42)  # for reproducibility

pipeline = take(
    format_logs(
        filter_by_level(
            generate_logs(10_000),  # 10k logs
            "ERROR"
        )
    ),
    5  # but only take 5
)

# Nothing has executed yet! The pipeline is lazy.
# Values flow through only when we consume them:
for entry in pipeline:
    print(entry)

# Memory: only one log dict exists at a time, regardless of input size.

### Exercise 3: Create a Context Manager for Database Connections

Create a context manager `DatabaseConnection` that:
1. Simulates opening a database connection on enter
2. Provides query execution methods
3. Automatically commits on clean exit
4. Automatically rolls back on exception
5. Always closes the connection

Implement it both as a class and using `@contextmanager`.

In [ ]:
# Exercise 3: Your solution here

# YOUR CODE HERE
pass

In [ ]:
# Exercise 3: Solution — Class-based context manager

class DatabaseConnection:
    """Simulated database connection with automatic commit/rollback."""
    
    def __init__(self, db_name: str):
        self.db_name = db_name
        self.connected = False
        self.queries: list[str] = []
    
    def connect(self):
        print(f"  Connecting to '{self.db_name}'...")
        self.connected = True
    
    def execute(self, query: str):
        if not self.connected:
            raise RuntimeError("Not connected!")
        print(f"  Executing: {query}")
        self.queries.append(query)
    
    def commit(self):
        print(f"  Committing {len(self.queries)} queries.")
        self.queries.clear()
    
    def rollback(self):
        print(f"  Rolling back {len(self.queries)} queries!")
        self.queries.clear()
    
    def close(self):
        print(f"  Closing connection to '{self.db_name}'.")
        self.connected = False
    
    def __enter__(self):
        self.connect()
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            self.rollback()
        else:
            self.commit()
        self.close()
        return False  # don't suppress exceptions

# Test: successful transaction
print("--- Successful transaction ---")
with DatabaseConnection("users.db") as db:
    db.execute("INSERT INTO users (name) VALUES ('Alice')")
    db.execute("INSERT INTO users (name) VALUES ('Bob')")

print()

# Test: failed transaction
print("--- Failed transaction ---")
try:
    with DatabaseConnection("users.db") as db:
        db.execute("INSERT INTO users (name) VALUES ('Charlie')")
        raise ValueError("Something went wrong!")  # simulate error
        db.execute("This should never run")
except ValueError as e:
    print(f"  Caught error: {e}")

In [ ]:
# Exercise 3: Solution — @contextmanager version

from contextlib import contextmanager

@contextmanager
def db_connection(db_name: str):
    """Context manager for simulated database connections."""
    db = DatabaseConnection(db_name)
    db.connect()
    try:
        yield db
        db.commit()  # only reached if no exception
    except Exception:
        db.rollback()
        raise  # re-raise the exception
    finally:
        db.close()  # always close

print("--- Using @contextmanager version ---")
with db_connection("products.db") as db:
    db.execute("SELECT * FROM products")

### Exercise 4: Retry Decorator with Exponential Backoff

Create a `@retry` decorator that:
1. Retries the function up to `max_retries` times on exception
2. Uses exponential backoff between retries (1s, 2s, 4s, ...)
3. Only retries on specified exception types
4. Logs each retry attempt
5. Raises the last exception if all retries fail

In [ ]:
# Exercise 4: Your solution here

# YOUR CODE HERE
pass

In [ ]:
# Exercise 4: Solution

import functools
import time

def retry(max_retries: int = 3, backoff_base: float = 1.0, exceptions: tuple = (Exception,)):
    """Decorator that retries a function with exponential backoff.
    
    Args:
        max_retries: Maximum number of retry attempts.
        backoff_base: Base delay in seconds (doubles each retry).
        exceptions: Tuple of exception types to catch.
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(max_retries + 1):  # +1 for initial attempt
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last_exception = e
                    if attempt < max_retries:
                        delay = backoff_base * (2 ** attempt)
                        print(f"  Attempt {attempt + 1} failed: {e}. "
                              f"Retrying in {delay:.1f}s...")
                        time.sleep(delay)
                    else:
                        print(f"  Attempt {attempt + 1} failed: {e}. "
                              f"No more retries.")
            raise last_exception
        return wrapper
    return decorator

# Test: function that fails intermittently
call_count = 0

@retry(max_retries=3, backoff_base=0.01, exceptions=(ConnectionError, TimeoutError))
def unreliable_api_call():
    """Simulates an API call that fails the first 2 times."""
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionError(f"Connection refused (attempt {call_count})")
    return {"status": "success", "data": [1, 2, 3]}

print("Calling unreliable API...")
result = unreliable_api_call()
print(f"Result: {result}")

### Exercise 5: Generic Type-Safe Container

Create a type-annotated `Result` class (inspired by Swift's `Result<Success, Failure>`) that:
1. Has `success` and `failure` class methods to construct instances
2. Has `map`, `flat_map` (like Swift's `map` and `flatMap` on Result)
3. Has `get_or_default` and `get_or_raise`
4. Is fully type-hinted using generics

In [ ]:
# Exercise 5: Your solution here

# YOUR CODE HERE
pass

In [ ]:
# Exercise 5: Solution

from typing import TypeVar, Generic, Callable
from dataclasses import dataclass

S = TypeVar('S')  # Success type
F = TypeVar('F', bound=Exception)  # Failure type (must be an Exception)
U = TypeVar('U')  # Mapped success type

class Result(Generic[S, F]):
    """A Result type inspired by Swift's Result<Success, Failure>.
    
    Swift equivalent:
        enum Result<Success, Failure: Error> {
            case success(Success)
            case failure(Failure)
        }
    """
    
    def __init__(self, value: S | None = None, error: F | None = None):
        if value is not None and error is not None:
            raise ValueError("Cannot have both value and error")
        if value is None and error is None:
            raise ValueError("Must have either value or error")
        self._value = value
        self._error = error
    
    @classmethod
    def success(cls, value: S) -> "Result[S, F]":
        return cls(value=value)
    
    @classmethod
    def failure(cls, error: F) -> "Result[S, F]":
        return cls(error=error)
    
    @property
    def is_success(self) -> bool:
        return self._value is not None
    
    @property
    def is_failure(self) -> bool:
        return self._error is not None
    
    def map(self, transform: Callable[[S], U]) -> "Result[U, F]":
        """Transform the success value, like Swift's Result.map."""
        if self.is_success:
            return Result.success(transform(self._value))
        return Result.failure(self._error)
    
    def flat_map(self, transform: Callable[[S], "Result[U, F]"]) -> "Result[U, F]":
        """Transform the success value with a function that returns a Result.
        Like Swift's Result.flatMap."""
        if self.is_success:
            return transform(self._value)
        return Result.failure(self._error)
    
    def get_or_default(self, default: S) -> S:
        """Return the success value or a default."""
        return self._value if self.is_success else default
    
    def get_or_raise(self) -> S:
        """Return the success value or raise the error."""
        if self.is_success:
            return self._value
        raise self._error
    
    def __repr__(self) -> str:
        if self.is_success:
            return f"Result.success({self._value!r})"
        return f"Result.failure({self._error!r})"

# Test it
def parse_int(s: str) -> Result[int, ValueError]:
    """Parse a string to int, returning a Result."""
    try:
        return Result.success(int(s))
    except ValueError as e:
        return Result.failure(e)

# Success case
r1 = parse_int("42")
print(r1)                          # Result.success(42)
print(r1.map(lambda x: x * 2))    # Result.success(84)
print(r1.get_or_default(0))        # 42

# Failure case
r2 = parse_int("not_a_number")
print(r2)                          # Result.failure(ValueError(...))
print(r2.map(lambda x: x * 2))    # Result.failure(ValueError(...))
print(r2.get_or_default(0))        # 0

# Chaining with flat_map
def double_if_positive(x: int) -> Result[int, ValueError]:
    if x > 0:
        return Result.success(x * 2)
    return Result.failure(ValueError(f"{x} is not positive"))

r3 = parse_int("21").flat_map(double_if_positive)
print(r3)  # Result.success(42)

r4 = parse_int("-5").flat_map(double_if_positive)
print(r4)  # Result.failure(ValueError('-5 is not positive'))

---

## Key Takeaways

1. **First-class functions** work the same in both languages, but Python's lambdas are more limited (single expression only).

2. **Decorators** are unique to Python. They are simply functions that wrap other functions. Always use `@functools.wraps`. Think of them as a composable way to add cross-cutting concerns (logging, timing, retrying, caching).

3. **Generators** replace the boilerplate of Swift's `Sequence`/`IteratorProtocol` with a simple `yield` keyword. Use them for memory-efficient processing of large datasets.

4. **Context managers** are more structured than Swift's `defer`. The `with` statement guarantees cleanup. Use `@contextmanager` for simple cases, class-based for complex ones.

5. **Type hints** give you the expressiveness of Swift's type system, but enforcement is optional and happens through external tools (`mypy`), not the runtime. Always add type hints in production code.

---

*Next up: Notebook 5 — File I/O and Error Handling*